In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.628000                    0.484300   
             precision                   0.540096                    0.364948   
             recall                      0.682461                    0.383695   
             f1                          0.601917                    0.373225   
             kappa                       0.261875                   -0.064131   
             MCC                         0.269131                   -0.064405   
outputsTest  accuracy                    0.631600                    0.484700   
             precision                   0.542105                    0.362697   
             recall                      0.687488                    0.392535   
             f1                          0.605437                    0.376119   
             kappa                       0.269045                   -0.061637   
             MCC                         0.276363                   -0.061869   
outputsAll   accuracy                    0.625600                    0.481800   
             precision                   0.538214                    0.364978   
             recall                      0.684526                    0.393437   
             f1                          0.601815                    0.377496   
             kappa                       0.258119                   -0.064667   
             MCC                         0.265232                   -0.065013   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.593800   
             precision                         0.651935   
             recall                            0.640628   
             f1                                0.645418   
             kappa                             0.169273   
             MCC                               0.169915   
outputsTest  accuracy                          0.591000   
             precision                         0.650700   
             recall                            0.634039   
             f1                                0.641378   
             kappa                             0.164747   
             MCC                               0.165430   
outputsAll   accuracy                          0.599900   
             precision                         0.657549   
             recall                            0.641889   
             f1                                0.648792   
             kappa                             0.183627   
             MCC                               0.184262   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.567400   
             precision                      0.423216   
             recall                         0.633663   
             f1                             0.506369   
             kappa                          0.148061   
             MCC                            0.158148   
outputsTest  accuracy                       0.560300   
             precision                      0.422671   
             recall                         0.643896   
             f1                             0.509184   
             kappa                          0.140972   
             MCC                            0.152030   
outputsAll   accuracy                       0.565100   
             precision                      0.418040   
             recall                         0.628755   
             f1                             0.500969   
             kappa                          0.142844   
             MCC                            0.152987   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.463200                  0.467600  
             precision                  0.215625                  0.136316  
             recall                     0.566399                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.270242,-0.063763,0.173202,0.154389,0.002307,-0.126689
accuracy,0.628400,0.483600,0.594900,0.564267,0.463133,0.464767
f1,0.603056,0.375613,0.645196,0.505508,0.312952,0.193156
kappa,0.263013,-0.063478,0.172549,0.143959,0.001264,-0.101185
precision,0.540138,0.364208,0.653394,0.421309,0.216925,0.135141
recall,0.684825,0.389889,0.638852,0.635438,0.569347,0.344864


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.307333                    0.012807   
             spearman                   0.308855                    0.015457   
             MSE                        1.385335                    1.974385   
             RMSE                       1.174669                    1.403574   
             MAE                        0.939994                    1.101678   
outputsTest  pearson                    0.286046                    0.008981   
             spearman                   0.291537                    0.025983   
             MSE                        1.427908                    1.982037   
             RMSE                       1.193061                    1.406502   
             MAE                        0.939491                    1.094584   
outputsAll   pearson                    0.302471                   -0.004338   
             spearman                   0.301722                    0.006044   
             MSE                        1.395058                    2.008677   
             RMSE                       1.179231                    1.415843   
             MAE                        0.939004                    1.108250   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.123981   
             spearman                         0.201191   
             MSE                              1.752038   
             RMSE                             1.321878   
             MAE                              1.002781   
outputsTest  pearson                          0.111456   
             spearman                         0.202900   
             MSE                              1.777089   
             RMSE                             1.331080   
             MAE                              1.006600   
outputsAll   pearson                          0.131665   
             spearman                         0.223700   
             MSE                              1.736670   
             RMSE                             1.315694   
             MAE                              0.991870   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.155734                 -0.005349   
             spearman                      0.173862                 -0.033165   
             MSE                           1.688532                  2.010698   
             RMSE                          1.297395                  1.415968   
             MAE                           0.991912                  1.115614   
outputsTest  pearson                       0.157807                  0.013736   
             spearman                      0.159350                 -0.004943   
             MSE                           1.684385                  1.972527   
             RMSE                          1.296019                  1.402674   
             MAE                           0.998303                  1.104421   
outputsAll   pearson                       0.163608                 -0.010131   
             spearman                      0.164684                 -0.038301   
             MSE                           1.672784                  2.020262   
             RMSE                          1.291751                  1.419599   
             MAE                           0.993236                  1.113477   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.062114  
             spearman                 -0.126155  
             MSE                       2.124228  
             RMSE                      1.455964  
             MAE                       1.077129  
outputsTest  pearson                  -0.064026  
             spearman                 -0.136934  
             MSE                       2.128053  
             RMSE                      1.457624  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.939496,1.101504,1.000417,0.994484,1.111171,1.084204
MSE,1.402767,1.988366,1.755266,1.681900,2.001162,2.120987
RMSE,1.182320,1.408640,1.322884,1.295055,1.412747,1.454962
pearson,0.298617,0.005817,0.122367,0.159050,-0.000581,-0.060493
spearman,0.300705,0.015828,0.209264,0.165965,-0.025470,-0.128139
